# IBOR → RFR Transition — Difference-in-Differences on Yield-Curve Parameters

Six of the fourteen universe currencies switched their swap curve from an IBOR
reference to a near risk-free rate (RFR) on a known cutover date (see
`data/raw/Zero Rates Data/README.md`). We ask whether that transition shifted
the fitted yield-curve **level** and **slope**
(`src/factors/nonstyle/yield_curve.py`) using a difference-in-differences (DiD)
design. As a robustness check we run the DiD under **two** ways of producing
those parameters — the Nelson-Siegel fit and a simple two-tenor construction
(120M rate as level, 24M minus 120M as slope) — and compare them. Curvature is
not used elsewhere in the project, so it is omitted.

**Variables.** `Treat` = 1 for a currency that transitioned, 0 otherwise;
`Post` = 1 on/after the cutover date, 0 before; `interaction` = `Treat` × `Post`.
The **interaction coefficient is the DiD estimate** — the average change in the
parameter for transitioned currencies around the cutover, net of the
contemporaneous change in the never-transitioned controls.

**Staggered timing.** The cutover dates differ (2022, 2023, 2024), so we split
the treated group into **cohorts** that share a date and estimate one DiD frame
per cohort. Each frame pairs its treated currencies with **all** never-treated
controls over a symmetric window around that cohort's cutover.

**Estimator.** Each frame is estimated with **two-way (entity + time) fixed
effects** via `PanelOLS`. The currency (entity) FE absorbs `Treat` and the date
(time) FE absorbs `Post`, so only `interaction` is passed to `exog`; that single
coefficient is the DiD effect. Estimated separately for level and slope, under
each of the two curve-parameter methods.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from linearmodels.panel import PanelOLS

from src import config
from src.data import loaders
from src.factors.nonstyle.yield_curve import fit_nelson_siegel, fit_simple_level_slope

## 1. Load the zero curve and fit both parameter methods

Each fitter returns one row per `date` × `currency` — already the panel the DiD
needs, no reshaping required. `fit_nelson_siegel` gives model-based `level` /
`slope` (plus an unused `curvature`); `fit_simple_level_slope` gives the raw
two-tenor `level` / `slope`. Both series run continuously across each cutover
(the old IBOR curve feeds them before, the new RFR curve after), so any break at
the cutover is exactly the effect we are after.

In [ ]:
cfg = config.load(PROJECT_ROOT / "config.yaml")
# Anchor the (relative) data root to the project root so the notebook works
# regardless of the directory the kernel is launched from.
cfg.raw["data"]["root"] = str(PROJECT_ROOT / cfg["data"]["root"])

zero_curve = loaders.load_table("zero_curve", cfg)


def to_panel(params):
    """Fitted (date, currency) params -> pandas panel keyed for PanelOLS."""
    p = params.to_pandas()
    p["date"] = pd.to_datetime(p["date"])
    p["currency"] = p["currency"].astype(str)
    return p


# One panel per curve-parameter method, compared head-to-head below.
panels = {
    "nelson_siegel": to_panel(fit_nelson_siegel(zero_curve, cfg)),
    "simple": to_panel(fit_simple_level_slope(zero_curve, cfg)),
}
panels["simple"].head()

## 2. Design: treatment groups, cutover dates, cohorts

Treated currencies and their cutover (first date of the new RFR curve, per the
README) are grouped into cohorts that share a date. Controls are the eight
currencies with a single curve throughout. `WINDOW` is the symmetric event
window around each cutover.

In [ ]:
# Treated: currency -> cutover date (first date of the new RFR-based curve).
TRANSITION_DATES = {
    "GBP": "2022-01-03", "CHF": "2022-01-03", "JPY": "2022-01-03",
    "USD": "2023-07-03", "SGD": "2023-07-03", "CAD": "2024-07-01",
}

# Never-treated controls: a single curve throughout.
CONTROLS = ["AUD", "HKD", "KRW", "TWD", "EUR", "DKK", "NOK", "SEK"]

# Treated currencies sharing a cutover form a cohort, so each DiD frame has one
# unambiguous event date shared by its treated and control units.
COHORTS = {
    pd.Timestamp("2022-01-03"): ["GBP", "CHF", "JPY"],
    pd.Timestamp("2023-07-03"): ["USD", "SGD"],
    pd.Timestamp("2024-07-01"): ["CAD"],
}

WINDOW = pd.DateOffset(months=24)   # symmetric window around each cutover
OUTCOMES = ["level", "slope"]

## 3. Build one DiD frame per method × cohort

Each frame keeps the cohort's treated currencies plus all controls, restricted
to ±`WINDOW` around the cutover. `Treat`, `Post` and `interaction` are attached,
and the frame is indexed by `(currency, date)` — the `(entity, time)` MultiIndex
`PanelOLS` expects. Frames are built for both parameter methods. The balance
check prints observation counts and the pre/post and treated/control shares per
cohort (reported for the Nelson-Siegel panel).

In [ ]:
def build_did_frame(panel, event_date, treated):
    """DiD frame for one cohort: treated + controls within +/- WINDOW of the cutover."""
    keep = set(treated) | set(CONTROLS)
    lo, hi = event_date - WINDOW, event_date + WINDOW
    frame = panel[panel["currency"].isin(keep) & panel["date"].between(lo, hi)].copy()
    frame["treat"] = frame["currency"].isin(treated).astype(int)
    frame["post"] = (frame["date"] >= event_date).astype(int)
    frame["interaction"] = frame["treat"] * frame["post"]
    return frame.set_index(["currency", "date"]).sort_index()


# One set of cohort frames per method: {method: {cohort_date: frame}}.
did_frames = {
    method: {d: build_did_frame(p, d, t) for d, t in COHORTS.items()}
    for method, p in panels.items()
}

for d, f in did_frames["nelson_siegel"].items():
    n_ent = f.index.get_level_values("currency").nunique()
    print(f"{d.date()}: {f.shape[0]} rows, {n_ent} currencies, "
          f"post share {f['post'].mean():.2f}, treated share {f['treat'].mean():.2f}")

## 4. Two-way fixed-effects DiD

For each method, cohort frame and outcome:

$$y_{it} = \alpha_i + \gamma_t + \delta\,(\text{Treat}_i \times \text{Post}_{it}) + \varepsilon_{it}$$

with currency fixed effects $\alpha_i$ (`entity_effects`) and date fixed effects
$\gamma_t$ (`time_effects`). The entity FE absorbs the time-invariant `Treat` and
the time FE absorbs the entity-invariant `Post`, so both are omitted from `exog`
and only `interaction` is estimated — its coefficient $\delta$ is the DiD effect.
Standard errors are clustered by currency.

> **Caveat:** each frame has only ~9–11 currency clusters, well below the usual
> threshold for reliable cluster-robust inference; read the p-values as
> indicative rather than exact.

In [ ]:
def run_twfe(frame, outcome):
    """TWFE DiD: y ~ interaction + EntityFE(currency) + TimeFE(date)."""
    mod = PanelOLS(
        frame[outcome],
        frame[["interaction"]],
        entity_effects=True,
        time_effects=True,
    )
    return mod.fit(cov_type="clustered", cluster_entity=True)


records = []
for method, frames in did_frames.items():
    for d, f in frames.items():
        for y in OUTCOMES:
            res = run_twfe(f, y)
            records.append({
                "method": method,
                "cohort": d.date(),
                "treated": ",".join(COHORTS[d]),
                "outcome": y,
                "did_coef": res.params["interaction"],
                "std_error": res.std_errors["interaction"],
                "t_stat": res.tstats["interaction"],
                "p_value": res.pvalues["interaction"],
                "n_obs": int(res.nobs),
            })

results = pd.DataFrame.from_records(records)
results

## 5. Results — Nelson-Siegel vs simple

`did_coef` is the estimated shift in each parameter (in the same decimal units as
`zero_rate`) attributable to the IBOR→RFR transition for that cohort. The pivot
below puts the two methods side by side for each cohort × outcome, so a robust
effect is one that agrees in sign and rough magnitude across methods.

In [ ]:
results.pivot_table(
    index=["cohort", "treated", "outcome"], columns="method", values="did_coef"
)

## Caveats

* **Few clusters.** Each cohort has ~9–11 currencies; cluster-robust standard
  errors are approximate at this cluster count.
* **Parallel trends.** DiD identification assumes treated and control curves
  would have moved together absent the transition. This is not tested here (an
  event-study on relative-time leads/lags is the natural next step).
* **Mechanical repricing.** The reference rate itself changes at the cutover
  (IBOR carries credit/term premia over a near-RFR), so a nonzero DiD partly
  reflects that repricing, not only curve-shape dynamics.
* **Fixed `tau`.** The NS decay is held fixed; the CAD post-window is truncated
  by the data end (2026-06-11).